# [2026-07-07] REPLACE WHERE flows are now generally available — 検証ノートブック

## このノートブックについて

Zenn 記事「[[2026-07-07] REPLACE WHERE flows are now generally available](https://zenn.dev/gtk0326/articles/d9-replace-where-flows-are-now-generally-available)」のハンズオン検証コードです。

記事と同じ手順を自分の Databricks 環境で再現できます。

> **注意**: SQL ウェアハウス（または SQL 対応クラスター）にアタッチし、各セルを上から順に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## セットアップ

セットアップ（ソーステーブルの作成・初期データ投入）


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.rwdemo;

-- ソース: 補正後の最新トランザクションが入る想定
CREATE OR REPLACE TABLE workspace.rwdemo.bronze (
  txn_date DATE, txn_id STRING, region STRING, amount DECIMAL(10,2)
);

-- 前日まで（8/2 まで）に確定していたデータ
INSERT INTO workspace.rwdemo.bronze VALUES
  (DATE '2026-07-28', 'T-100', 'east', 50.00),  -- 直近3日より前（ウィンドウ外）
  (DATE '2026-07-31', 'T-201', 'east', 60.00),
  (DATE '2026-08-01', 'T-202', 'west', 30.00),
  (DATE '2026-08-02', 'T-203', 'east', 20.00);

## ステップ1: REPLACE WHERE フローを Lakeflow パイプラインで定義する

置き換え先のストリーミングテーブルを宣言し、そこへ流し込むフローを書きます。述語は `date_add(current_date(), -3)` で「直近 3 日」を表します。


In [ ]:
%sql
-- パイプラインのターゲット
CREATE OR REFRESH STREAMING TABLE workspace.rwdemo_out.silver;

-- 「直近3日ぶん」を毎回まるごと置き換えるフロー
CREATE FLOW backfill_recent
AS INSERT INTO workspace.rwdemo_out.silver BY NAME
REPLACE WHERE txn_date >= date_add(current_date(), -3)
SELECT txn_date, txn_id, region, amount
FROM workspace.rwdemo.bronze
WHERE txn_date >= date_add(current_date(), -3);

## ステップ2: 1回目のビルド結果を確認する（Before）

パイプラインを実行すると、イベントログに置き換えモードが記録されます。
実行結果（パイプラインイベントログ抜粋）:
```
Flow 'backfill_recent' has been planned to be executed as COMPLETE_REPLACE_WHERE.
Flow 'backfill_recent' has COMPLETED.
Update 32d928 is COMPLETED.
```
この時点の `silver` を見てみます。


In [ ]:
%sql
SELECT * FROM workspace.rwdemo_out.silver ORDER BY txn_date, txn_id;

## ステップ3: 補正がソースに届く

翌日の運用を想定して、`bronze` に返品・訂正・遅延到着を反映します。


In [ ]:
%sql
-- T-201: 返品で 60.00 → 40.00 に減額
UPDATE workspace.rwdemo.bronze SET amount = 40.00 WHERE txn_id = 'T-201';
-- T-203: 入力訂正で 20.00 → 25.00
UPDATE workspace.rwdemo.bronze SET amount = 25.00 WHERE txn_id = 'T-203';
-- T-204: 8/3 ぶんが遅れて届いた（新規）
INSERT INTO workspace.rwdemo.bronze VALUES (DATE '2026-08-03', 'T-204', 'west', 12.00);

## ステップ4: 2回目のビルド結果を確認する（After）

同じパイプラインをもう一度実行し、`silver` を見ます。


In [ ]:
%sql
SELECT * FROM workspace.rwdemo_out.silver ORDER BY txn_date, txn_id;

## これ、MERGE でもできるのでは？

ここまで読んで、「同じことは既存の MERGE でも書けるのでは」と思った方も多いはずです。実際そのとおりで、機能としては MERGE でも実現できます。念のため同じ補正取り込みを MERGE で書くと、こうなります。


In [ ]:
%sql
MERGE INTO workspace.rwdemo.silver_merge AS t
USING (
  SELECT txn_date, txn_id, region, amount
  FROM workspace.rwdemo.bronze
  WHERE txn_date >= date_add(current_date(), -3)
) AS s
ON t.txn_id = s.txn_id AND t.txn_date >= date_add(current_date(), -3)
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE AND t.txn_date >= date_add(current_date(), -3) THEN DELETE;

## クリーンアップ

検証で作成したオブジェクトをすべて削除します。

> **必ず実行してください。** ストリーミングテーブルやマテリアライズドビューを残すとバックグラウンドで計算リソースを消費し続けます。

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.rwdemo_out.silver;
DROP TABLE IF EXISTS workspace.rwdemo.bronze;

-- スキーマを削除（内包するオブジェクトもすべて削除）
DROP SCHEMA IF EXISTS workspace.rwdemo CASCADE;